In [7]:
import pandas as pd
import numpy as np
import idx2numpy

In [8]:
# read data
X_train = idx2numpy.convert_from_file('data/mnist/train-images.idx3-ubyte')
y_train = idx2numpy.convert_from_file('data/mnist/train-labels.idx1-ubyte')

X_train = (X_train.reshape(X_train.shape[0], -1) / 255.0).T
y_train = y_train.reshape(1, -1)

print("shape:")
print("X_train -", X_train.shape)
print("y_train -", y_train.shape)

shape:
X_train - (784, 60000)
y_train - (1, 60000)


In [9]:
def ReLU(Z):
    return np.maximum(Z, 0)

def ReLU_deriv(Z):
    return Z > 0

def softmax(Z):
    return np.exp(Z) / sum(np.exp(Z))

def init_params():
    W1 = np.random.rand(128, 784)
    b1 = np.random.rand(128, 1)
    W2 = np.random.rand(10, 128)
    b2 = np.random.rand(10, 1)

    return W1, b1, W2, b2

def forward_prop(x, W1, b1, W2, b2):
    z1 = W1.dot(x) + b1
    a1 = ReLU(z1)
    z2 = W2.dot(a1) + b2
    a2 = softmax(z2)

    return z1, a1, z2, a2

def one_hot_encode(y):
    one_hot_y = np.zeros((y.size, 10))
    one_hot_y[np.arange(y.size), y] = 1
    one_hot_y = one_hot_y.T
    return one_hot_y.T

def back_prop(z1, a1, a2, W2, X, y):
    m = X.shape[1]
    one_hot_y = one_hot_encode(y)
    dz2 = a2 - one_hot_y
    dW2 = (1 / m) * dz2.dot(a1.T)
    db2 = (1 / m) * a2 - y
    dz1 = W2.T.dot(dz2) * ReLU_deriv(z1)
    dW1 = (1 / m) * dz1.dot(X.T)
    db1 = (1 / m) * np.sum(dz1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2

    return W1, b1, W2, b2

def get_predictions(a2):
    return np.argmax(a2, axis=0)

def get_accuracy(predictions, y):
    return np.sum(predictions == y) / y.size

def mlp_train(x, y, iterations, alpha):
    W1, b1, W2, b2 = init_params()

    for i in range(iterations):
        z1, a1, z2, a2 = forward_prop(x, W1, b1, W2, b2)
        dW1, db1, dW2, db2 = back_prop(z1, a1, a2, W2, x, y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)

        if i % 10 == 0 or i == iterations - 1:
            predictions = get_predictions(a2)
            acc = get_accuracy(predictions, y)
            print(f"Iteration {iterations} | acc: {acc}")

    return W1, b1, W2, b2

In [11]:
# 3 layers - 1st input (28 x 28) - 2nd hidden (128) - 3rd output (10)
W1, b1, W2, b2 = mlp_train(X_train, y_train, iterations=200, alpha=0.5)

/var/folders/tz/63wnq6092dbbvv9gwhb3j8400000gn/T/ipykernel_9721/2505930759.py:8: RuntimeWarning: overflow encountered in exp
  return np.exp(Z) / sum(np.exp(Z))
/var/folders/tz/63wnq6092dbbvv9gwhb3j8400000gn/T/ipykernel_9721/2505930759.py:8: RuntimeWarning: invalid value encountered in divide
  return np.exp(Z) / sum(np.exp(Z))


ValueError: operands could not be broadcast together with shapes (10,60000) (60000,10) 